In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
def encodeLabel(data, feature):
    encoder = LabelEncoder()
    data[feature] = encoder.fit_transform(data[feature].fillna('none'))
    return data

def jaccard(cm, coupon):
    cm = set(cm)
    coupon = set(coupon)
    return len(cm & coupon) / len(cm | coupon)

In [3]:
driver = pd.read_csv('../data/driver.csv')

In [4]:
item = pd.read_csv('../data/data/item_data.csv')
item = encodeLabel(item, 'brand')
item = encodeLabel(item, 'brand_type')
item = encodeLabel(item, 'category')

In [5]:
tranx = pd.read_csv('../data/data/customer_transaction_data.csv')
tranx = tranx[tranx['coupon_discount'] == 0]
tranx = tranx.merge(item, on='item_id')

In [6]:
coupon = pd.read_csv('../data/data/coupon_item_mapping.csv')
coupon = coupon.merge(item, on='item_id')

In [7]:
cust_item = tranx.groupby('customer_id')['item_id'].apply(list).reset_index().rename(columns={'item_id':'cm_ilist'})
coup_item = coupon.groupby('coupon_id')['item_id'].apply(list).reset_index().rename(columns={'item_id':'cp_ilist'})

In [8]:
cust_brand = tranx.groupby('customer_id')['brand'].apply(list).reset_index().rename(columns={'brand':'cm_blist'})
coup_brand = coupon.groupby('coupon_id')['brand'].apply(list).reset_index().rename(columns={'brand':'cp_blist'})

In [9]:
cust_cat = tranx.groupby('customer_id')['category'].apply(list).reset_index().rename(columns={'category':'cm_clist'})
coup_cat = coupon.groupby('coupon_id')['category'].apply(list).reset_index().rename(columns={'category':'cp_clist'})

In [10]:
driver = driver.merge(cust_item, on=['customer_id'])
driver = driver.merge(coup_item, on=['coupon_id'])
driver = driver.merge(cust_brand, on=['customer_id'])
driver = driver.merge(coup_brand, on=['coupon_id'])
driver = driver.merge(cust_cat, on=['customer_id'])
driver = driver.merge(coup_cat, on=['coupon_id'])

In [11]:
driver['over_1'] = driver[['cm_ilist','cp_ilist']].apply(lambda x : jaccard(*x), axis=1)
driver['over_2'] = driver[['cm_blist','cp_blist']].apply(lambda x : jaccard(*x), axis=1)
driver['over_3'] = driver[['cm_clist','cp_clist']].apply(lambda x : jaccard(*x), axis=1)

In [12]:
driver = driver[['id','over_1','over_2','over_3']]

In [13]:
driver.head()

,id,over_1,over_2,over_3
0,1,0.000000,0.000000,0.125000
1,111151,0.000000,0.006250,0.083333
2,33428,0.033771,0.017699,0.125000
3,45518,0.000000,0.012658,0.083333
4,111005,0.000000,0.000000,0.083333


In [14]:
driver.to_csv('../data/feature/similarity.csv', index=False)